In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.datasets import mnist

# Load Dataset

(x_train, y_train), (x_test, y_test) = mnist.load_data()

digit = int(input("Enter a digit (0-9): "))

if 0<=digit<=9:
  indexes = np.where(y_train == digit)[0][:5]

  plt.figure(figsize=(5, 2))

  for i, index in enumerate(indexes):
    plt.subplot(1, 5, i+1)
    plt.imshow(x_train[index], cmap='gray')
    plt.axis('off')
    plt.title(digit)

  plt.suptitle(f"First 5 Images of the Digit {digit}")
  plt.show()

else:
  print("Invalid Input!")

In [ ]:
from keras.datasets.cifar10 import load_data
import matplotlib.pyplot as plt
(x_train,y_train),(x_test,y_test)=load_data()

for i in range(49):
  plt.subplot(7,7,i+1)
  plt.axis("off")
  plt.imshow(x_train[i])
plt.show()

149913600/170498071 ━━━━━━━━━━━━━━━━━━━━ 8:13 24us/step

In [ ]:
print(x_train.shape)

# Libraries for Discriminator

In [ ]:
from keras.models import Sequential
from keras.layers import Dense,Dropout,Flatten,Reshape,LeakyReLU
from keras.layers import Conv2D
from keras.optimizers import Adam
from keras.utils import plot_model
import numpy as np

In [ ]:
def define_discriminator(in_shape=(32,32,3),lr=0.0002):
   model=Sequential()
   model.add(Conv2D(64,(3,3),padding='same',input_shape=in_shape))
   model.add(LeakyReLU(alpha=0.2))

   model.add(Conv2D(128,(3,3),strides=(2,2),padding='same'))
   model.add(LeakyReLU(alpha=0.2))

   model.add(Conv2D(128,(3,3),strides=(2,2),padding='same'))
   model.add(LeakyReLU(alpha=0.2))

   model.add(Conv2D(256,(3,3),strides=(2,2),padding='same'))
   model.add(LeakyReLU(alpha=0.2))

   model.add(Flatten())
   model.add(Dropout(0.4))
   model.add(Dense(1,activation='sigmoid'))

   opt=Adam(lr,beta_1=0.5)
   model.compile(loss='binary_crossentropy',optimizer=opt,metrics=['accuracy'])
   return model

In [ ]:
model=define_discriminator()
model.summary()
plot_model(model,to_file='discriminator_plot.png',show_shapes=True,show_layer_names=True)

In [ ]:
def load_real_samples():
  (x_train,_),(_,_)=load_data()
  x=x_train.astype('float32')
  x=(x-127.5)/127.5
  return x_train

In [ ]:
X=load_real_samples()
print(X.shape)

In [ ]:
def generate_real_samples(dataset,n_samples):
  ix=np.random.randint(0,dataset.shape[0],n_samples)
  X=dataset[ix]
  y=np.ones((n_samples,1))
  return X,y

In [ ]:
X,y=generate_real_samples(X,64)
print(X.shape,y.shape)
print(y)

In [ ]:
def generate_fake_samples(n_samples):
  X=np.random.rand(n_samples * 32 * 32 * 3)
  X=-1+X*2
  X=X.reshape(n_samples,32,32,3)
  y=np.zeros((n_samples,1))
  return X,y

In [ ]:
X,y=generate_fake_samples(64)
print(X.shape,y.shape)
print(y)

In [ ]:
plt.imshow(X[0])
plt.show()

In [ ]:
def  train_discriminator(model,dataset,n_iter=20,n_batch=128):
  half_batch=int(n_batch/2)
  for i in range(n_iter):
    X_real,Y_real=generate_real_samples(dataset,half_batch)
    _,real_acc=model.train_on_batch(X_real,Y_real)

    X_fake,Y_fake=generate_fake_samples(half_batch)
    _,fake_acc=model.train_on_batch(X_fake,Y_fake)

    print('>%d real=%0.0f%% fake=%0.0f%%'%(i+1,real_acc*100,fake_acc*100))

In [ ]:
model=define_discriminator()
dataset=load_real_samples()
train_discriminator(model,dataset)

# GENERATOR PART

In [ ]:
from keras.models import Sequential
from keras.layers import Dense,Reshape,LeakyReLU
from keras.layers import Conv2D,Conv2DTranspose
from keras.utils import plot_model

In [ ]:
def define_generator(latent_dim):
  model=Sequential()
  n_nodes=256*4*4
  model.add(Dense(n_nodes,input_dim=latent_dim))
  model.add(LeakyReLU(alpha=0.2))
  model.add(Reshape((4,4,256)))

  model.add(Conv2DTranspose(128,(4,4),strides=(2,2),padding='same'))
  model.add(LeakyReLU(alpha=0.2))

  model.add(Conv2DTranspose(128,(4,4),strides=(2,2),padding='same'))
  model.add(LeakyReLU(alpha=0.2))

  model.add(Conv2DTranspose(128,(4,4),strides=(2,2),padding='same'))
  model.add(LeakyReLU(alpha=0.2))

  model.add(Conv2D(3,(3,3),activation='tanh',padding='same'))
  return model


In [ ]:
latent_dim=100
model=define_generator(latent_dim)
model.summary()
plot_model(model,to_file='generator_plot.png',show_shapes=True,show_layer_names=True)

In [ ]:
def generate_latent_points(latent_dim,n_samples):
  x_input=np.random.rand(latent_dim*n_samples)
  x_input=x_input.reshape(n_samples,latent_dim)
  return x_input

In [ ]:
x_input=generate_latent_points(100,64)
print(x_input.shape)


In [ ]:
def generate_fake_samples(g_model,latent_dim,n_samples):
  x_input=generate_latent_points(latent_dim,n_samples)
  X=g_model.predict(x_input)
  y=np.zeros((n_samples,1))
  return X,y

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
latent_dim=100
model=define_generator(latent_dim)
n_samples=49
X,_=generate_fake_samples(model,latent_dim,n_samples)
X=(X+1)/2.0
for i in range(n_samples):
  plt.subplot(7,7,i+1)
  plt.axis("off")
  plt.imshow(X[i])
plt.show()


In [ ]:
def define_gan(g_model,d_model):
  d_model.trainable=False
  model=Sequential()
  model.add(g_model)
  model.add(d_model)
  opt=Adam(lr=0.0002,beta_1=0.5)
  model.compile(loss='binary_crossentropy',optimizer=opt)
  return model

In [ ]:
latent_dim=100
d_model=define_discriminator()
g_model=define_generator(latent_dim)
gan_model=define_gan(g_model,d_model)
gan_model.summary()
plot_model(gan_model,to_file='gan_plot.png',show_shapes=True,show_layer_names=True)

In [ ]:
def train(g_model,d_model,gan_model,dataset,latent_dim,n_epochs=20,n_batch=128):
  bat_per_epo=int(dataset.shape[0]/n_batch)
  half_batch=int(n_batch/2)
  for i in range(n_epochs):
    for j in range(bat_per_epo):
      X_real,Y_real=generate_real_samples(dataset,half_batch)
      d_loss1,_=d_model.train_on_batch(X_real,Y_real)
      X_fake,Y_fake=generate_fake_samples(g_model,latent_dim,half_batch)
      d_loss2,_=d_model.train_on_batch(X_fake,Y_fake)
      X_gan=generate_latent_points(latent_dim,n_batch)
      y_gan=np.ones((n_batch,1))
      g_loss=gan_model.train_on_batch(X_gan,y_gan)
      print('>%d, %d/%d, d1=%.3f, d2=%.3f g=%.3f'%(i+1,j+1,bat_per_epo,d_loss1,d_loss2,g_loss))
    if (i+1)%10==0:
      summarize_performance(i,g_model,d_model,dataset,latent_dim)

In [ ]:
def summarize_performance(epoch,g_model,d_model,dataset,latent_dim,n_samples=150):
  X_real,y_real=generate_real_samples(dataset,n_samples)
  _,acc_real=d_model.evaluate(X_real,y_real,verbose=0)
  x_fake,y_fake=generate_fake_samples(g_model,latent_dim,n_samples)
  _,acc_fake=d_model.evaluate(x_fake,y_fake,verbose=0)
  print('>Accuracy real=%.0f%%,fake=%.0f%%'%(acc_real*100,acc_fake*100))
  save_plot(x_fake,epoch)
  filename='generator_model_%03d.h5'%(epoch+1)
  g_model.save(filename)

In [ ]:
def save_plot(examples,epoch,n=7):
  examples=(examples+1)/2.0
  for i in range(n*n):
    plt.subplot(n,n,i+1)
    plt.axis('off')
    plt.imshow(examples[i])
  filename='generated_plot_e%03d.png'%(epoch+1)
  plt.savefig(filename)
  plt.close()